# ZFN Backend Runtime Comparison

This notebook benchmarks the three live ZFN search backends on identical direct-Python inputs:

- `exhaustive_python`
- `pyahocorasick`
- `fm_index`

The notebook now supports two comparison tiers:

- a fast deterministic synthetic fixture for repeatable parity and timing checks
- an opt-in real Ensembl human chromosome 3 case with Ensembl GTF annotation enabled

Outputs include:

- per-run wall-clock and phase timings
- normalized backend-to-backend result equality checks
- runtime and phase-breakdown plots by case
- annotation-region summaries for annotated runs
- CSV and JSON artifacts under `notebooks/zfn_experiment_runs/backend_runtime_comparison`

The real chr3 case is intentionally opt-in because `exhaustive_python` remains materially more expensive at that scale, especially once annotation is added.

## 1. Import Required Libraries

Import the standard library modules, third-party packages, and `sirnaforge` internals needed for benchmarking the shared ZFN search path.

In [ ]:
from __future__ import annotations

import hashlib
import importlib
import json
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from Bio.Seq import Seq
from IPython.display import display

import sirnaforge.models.zfn as zfn_models
import sirnaforge.zfn.search as zfn_search

importlib.reload(zfn_models)
importlib.reload(zfn_search)

from sirnaforge.models.zfn import (
    GenomicAnnotationConfig,
    ZFNAlgorithm,
    ZFNDesignParameters,
    ZFNHalfSiteConstraints,
    ZFNShardingConfig,
    ZFNSearchBackend,
    ZFNSpacerConstraints,
)
from sirnaforge.zfn.rank import rank_sites
from sirnaforge.zfn.search import ExhaustiveZFNOffTargetSearcher, build_zfn_search_index

## 2. Define Configuration and Inputs

Declare the runtime switches, benchmark constants, and artifact paths used by the notebook. The default case is a deterministic synthetic multi-contig FASTA so the benchmark stays reproducible and local.

In [ ]:
NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "notebooks":
    NOTEBOOK_DIR = Path("/home/hovland/sirnaforge/sirnaforge/notebooks")
WORKSPACE_ROOT = NOTEBOOK_DIR.parent
RUN_ROOT = NOTEBOOK_DIR / "zfn_experiment_runs"
ARTIFACT_ROOT = RUN_ROOT / "backend_runtime_comparison"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
FIXTURE_ROOT = ARTIFACT_ROOT / "fixtures"
FIXTURE_ROOT.mkdir(parents=True, exist_ok=True)

LEFT_HALF_SITE = "GTCATCCTCATC"
RIGHT_HALF_SITE = "AAACTGCAAAAG"
ALLOWED_SPACERS = [5, 6]
SYNTHETIC_REPEATS = 3
REAL_CHR3_REPEATS = 1
TOP_N_SITES = 5000
SYNTHETIC_CHUNK_SIZE_BP = 250_000
REAL_CHR3_CHUNK_SIZE_BP = 12_000_000
MAX_WORKERS = 2
RUN_REAL_CHR3 = False
RUN_FM_INDEX_PERSISTED_APPENDIX = True
HG38_ANNOTATION_URL = "https://ftp.ensembl.org/pub/current_gtf/homo_sapiens/Homo_sapiens.GRCh38.115.gtf.gz"
HG38_CHR3_FASTA_URL = (
    "https://ftp.ensembl.org/pub/current_fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.chromosome.3.fa.gz"
)

LIVE_BACKENDS = [
    ZFNSearchBackend.EXHAUSTIVE_PYTHON,
    ZFNSearchBackend.PYAHOCORASICK,
    ZFNSearchBackend.FM_INDEX,
]
PHASE_COLUMNS = [
    "resolve_inputs_s",
    "load_fasta_s",
    "build_shards_s",
    "search_shards_s",
    "dedupe_s",
    "annotate_s",
    "rank_s",
    "truncate_s",
]
NORMALIZED_COLUMNS = [
    "chrom",
    "start_1based",
    "end_1based",
    "strand",
    "orientation",
    "spacer_len",
    "sequence",
    "left_mismatches",
    "right_mismatches",
    "total_mismatches",
    "score",
    "region",
    "nearest_gene",
]

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")

for module_name in ("ahocorasick", "fm_index"):
    if importlib.util.find_spec(module_name) is None:
        raise RuntimeError(
            f"Missing optional dependency '{module_name}'. Install the full ZFN backend extras before running this notebook."
        )

print(f"Workspace root: {WORKSPACE_ROOT}")
print(f"Artifact root: {ARTIFACT_ROOT}")
print(f"Live backends: {[backend.value for backend in LIVE_BACKENDS]}")
print(f"Real chr3 enabled: {RUN_REAL_CHR3}")
print(f"HG38 chr3 URL: {HG38_CHR3_FASTA_URL}")
print(f"HG38 annotation URL: {HG38_ANNOTATION_URL}")
print("Sharding defaults:", ZFNShardingConfig().model_dump())

## 3. Create Helper Utilities

Implement the deterministic fixture builder, result normalization helpers, and small formatting utilities that keep the benchmark logic compact and reproducible.

In [ ]:
def canonical_site(left: str, right: str, spacer: str = "AAAAA") -> str:
    return f"{left}{spacer}{str(Seq(right).reverse_complement())}"


def build_deterministic_multicontig_fasta(output_dir: Path) -> Path:
    chr2 = (
        ("ACGT" * 40_000)
        + canonical_site(LEFT_HALF_SITE, RIGHT_HALF_SITE, spacer="AAAAA")
        + ("TGCA" * 35_000)
        + canonical_site(LEFT_HALF_SITE, RIGHT_HALF_SITE, spacer="AAAAAA")
        + ("GATC" * 20_000)
    )
    chr3 = (
        ("CATG" * 30_000)
        + canonical_site(LEFT_HALF_SITE, RIGHT_HALF_SITE, spacer="AAAAA")
        + ("TTAA" * 25_000)
        + canonical_site(LEFT_HALF_SITE, RIGHT_HALF_SITE, spacer="AAAAAA")
        + ("CGCG" * 25_000)
    )
    fasta_path = output_dir / "zfn_backend_runtime_fixture.fa"
    fasta_path.write_text(f">chr2\n{chr2}\n>chr3\n{chr3}\n", encoding="utf-8")
    return fasta_path


def make_params(
    *,
    case: BenchmarkCase,
    backend: ZFNSearchBackend,
    search_space_index: str | None = None,
) -> ZFNDesignParameters:
    return ZFNDesignParameters(
        search_space_fasta=str(case.search_space_fasta),
        search_space_index=search_space_index,
        left_half_site=LEFT_HALF_SITE,
        right_half_site=RIGHT_HALF_SITE,
        search_backend=backend,
        algorithm=case.algorithm,
        top_n_sites=case.top_n_sites,
        half_site_constraints=ZFNHalfSiteConstraints(
            max_mismatches=case.max_mismatches,
            seed_len_from_fokI=case.seed_len_from_fokI,
            seed_max_mismatches=case.seed_max_mismatches,
            window_stride=1,
        ),
        spacer_constraints=ZFNSpacerConstraints(allowed_spacer_lengths=ALLOWED_SPACERS),
        sharding=ZFNShardingConfig(
            enabled=True,
            chunk_size_bp=case.chunk_size_bp,
            overlap_bp=50,
            chromosomes=[],
            max_workers=case.max_workers,
        ),
    )


def make_annotation(case: BenchmarkCase) -> GenomicAnnotationConfig | None:
    if case.annotation_reference is None:
        return None
    return GenomicAnnotationConfig(annotation_reference=case.annotation_reference)


def sites_to_frame(sites: list[Any]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for site in sites:
        score = float(site.score) if site.score is not None else float("nan")
        rows.append(
            {
                "chrom": site.chrom,
                "start_1based": int(site.start_1based),
                "end_1based": int(site.end_1based),
                "strand": site.strand.value if hasattr(site.strand, "value") else str(site.strand),
                "orientation": site.orientation.value if hasattr(site.orientation, "value") else str(site.orientation),
                "spacer_len": int(site.spacer_len),
                "sequence": site.sequence,
                "left_mismatches": int(site.left_mismatches),
                "right_mismatches": int(site.right_mismatches),
                "total_mismatches": int(site.total_mismatches),
                "score": round(score, 8),
                "region": site.region,
                "nearest_gene": site.nearest_gene,
            }
        )
    return normalize_site_frame(pd.DataFrame(rows, columns=NORMALIZED_COLUMNS))


def normalize_site_frame(df: pd.DataFrame) -> pd.DataFrame:
    normalized = df.copy()
    for column in [
        "start_1based",
        "end_1based",
        "spacer_len",
        "left_mismatches",
        "right_mismatches",
        "total_mismatches",
        "score",
    ]:
        normalized[column] = pd.to_numeric(normalized[column])
    normalized["score"] = normalized["score"].round(8)
    return normalized.sort_values(
        by=["chrom", "start_1based", "end_1based", "orientation", "total_mismatches", "sequence"]
    ).reset_index(drop=True)


def fingerprint_frame(df: pd.DataFrame) -> str:
    payload = df.to_csv(index=False).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()[:16]


def summarize_results(results: pd.DataFrame) -> pd.DataFrame:
    return (
        results.groupby(["case_id", "case_label", "annotation_enabled", "backend"], as_index=False)
        .agg(
            mean_wall_s=("wall_s", "mean"),
            std_wall_s=("wall_s", "std"),
            mean_search_shards_s=("search_shards_s", "mean"),
            mean_annotate_s=("annotate_s", "mean"),
            mean_rank_s=("rank_s", "mean"),
            mean_rows=("result_rows", "mean"),
            parity_all=("parity_ok", "all"),
        )
        .sort_values(["case_id", "mean_wall_s"])
        .reset_index(drop=True)
    )


def summarize_annotation_regions(
    normalized_frames: dict[str, pd.DataFrame],
    cases: list[BenchmarkCase],
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for case in cases:
        if case.annotation_reference is None:
            continue
        for backend in LIVE_BACKENDS:
            frame = normalized_frames[f"{case.case_id}::{backend.value}_repeat_1"]
            region_counts = frame["region"].value_counts().to_dict()
            rows.append(
                {
                    "case_id": case.case_id,
                    "backend": backend.value,
                    "rows": len(frame),
                    "exon": int(region_counts.get("exon", 0)),
                    "promoter": int(region_counts.get("promoter", 0)),
                    "intron": int(region_counts.get("intron", 0)),
                    "intergenic": int(region_counts.get("intergenic", 0)),
                    "unknown": int(region_counts.get("unknown", 0)),
                    "unique_nearest_genes": int(frame["nearest_gene"].dropna().nunique()),
                }
            )
    return pd.DataFrame(rows)

## 4. Implement Core Data Structures

Define the case and run containers that describe benchmark intent and preserve per-run metadata in a flat, dataframe-friendly format.

In [ ]:
@dataclass(frozen=True)
class BenchmarkCase:
    case_id: str
    label: str
    search_space_fasta: str | Path
    repeats: int
    top_n_sites: int
    max_mismatches: int
    seed_len_from_fokI: int | None
    seed_max_mismatches: int | None
    chunk_size_bp: int
    max_workers: int
    algorithm: ZFNAlgorithm = ZFNAlgorithm.HOMOLOGY
    annotation_reference: str | None = None


@dataclass(frozen=True)
class BenchmarkRun:
    case_id: str
    case_label: str
    backend: str
    repeat: int
    annotation_enabled: bool
    wall_s: float
    result_rows: int
    parity_ok: bool
    fingerprint: str
    resolved_fasta: str
    resolved_annotation: str | None
    shard_count: int
    effective_workers: int
    raw_sites: int
    deduped_sites: int
    ranked_sites: int
    retained_sites: int
    resolve_inputs_s: float
    load_fasta_s: float
    build_shards_s: float
    search_shards_s: float
    dedupe_s: float
    annotate_s: float
    rank_s: float
    truncate_s: float


fixture_fasta = build_deterministic_multicontig_fasta(FIXTURE_ROOT)
CASES = [
    BenchmarkCase(
        case_id="deterministic_multicontig",
        label="Deterministic synthetic multi-contig fixture",
        search_space_fasta=fixture_fasta,
        repeats=SYNTHETIC_REPEATS,
        top_n_sites=TOP_N_SITES,
        max_mismatches=0,
        seed_len_from_fokI=6,
        seed_max_mismatches=0,
        chunk_size_bp=SYNTHETIC_CHUNK_SIZE_BP,
        max_workers=MAX_WORKERS,
        algorithm=ZFNAlgorithm.HOMOLOGY,
    )
]
if RUN_REAL_CHR3:
    CASES.append(
        BenchmarkCase(
            case_id="hg38_chr3_annotated",
            label="Ensembl hg38 chr3 with Ensembl GTF annotation",
            search_space_fasta=HG38_CHR3_FASTA_URL,
            repeats=REAL_CHR3_REPEATS,
            top_n_sites=TOP_N_SITES,
            max_mismatches=2,
            seed_len_from_fokI=6,
            seed_max_mismatches=1,
            chunk_size_bp=REAL_CHR3_CHUNK_SIZE_BP,
            max_workers=MAX_WORKERS,
            algorithm=ZFNAlgorithm.ZFN_V2,
            annotation_reference=HG38_ANNOTATION_URL,
        )
    )

display(pd.DataFrame([asdict(case) for case in CASES]))

## 5. Implement Main Processing Logic

Build an instrumented searcher around the shared `sirnaforge` ZFN search path, run repeated backend comparisons, and produce compact summary and plotting helpers.

In [ ]:
class InstrumentedSearcher(ExhaustiveZFNOffTargetSearcher):
    def profile(
        self,
        params: ZFNDesignParameters,
        annotation: GenomicAnnotationConfig | None = None,
    ) -> dict[str, Any]:
        phase_timings = {phase: 0.0 for phase in PHASE_COLUMNS}
        started_at = time.perf_counter()

        phase_start = time.perf_counter()
        fasta_path = self._resolve_search_space_fasta(params)
        resolved_annotation: str | None = None
        if annotation is not None:
            annotation_path = self._resolve_annotation_path(annotation)
            resolved_annotation = str(annotation_path) if annotation_path is not None else None
        phase_timings["resolve_inputs_s"] = time.perf_counter() - phase_start

        phase_start = time.perf_counter()
        chrom_sequences = self._load_fasta(fasta_path)
        phase_timings["load_fasta_s"] = time.perf_counter() - phase_start

        phase_start = time.perf_counter()
        shard_specs = self._build_shard_specs(chrom_sequences, params)
        phase_timings["build_shards_s"] = time.perf_counter() - phase_start

        scan_engine = self._scan_engine_for(params)
        effective_workers = (
            min(self._recommended_worker_cap(chrom_sequences, params), len(shard_specs)) if shard_specs else 1
        )

        phase_start = time.perf_counter()
        raw_sites = self._run_shard_searches(
            shard_specs=shard_specs,
            chrom_sequences=chrom_sequences,
            params=params,
            scan_engine=scan_engine,
            workers=effective_workers,
            started_at=phase_start,
        )
        phase_timings["search_shards_s"] = time.perf_counter() - phase_start

        phase_start = time.perf_counter()
        deduped = self._dedupe_sites(raw_sites)
        phase_timings["dedupe_s"] = time.perf_counter() - phase_start

        if annotation is not None and self.annotation_provider is not None:
            phase_start = time.perf_counter()
            deduped = self.annotation_provider.annotate(deduped, annotation)
            phase_timings["annotate_s"] = time.perf_counter() - phase_start

        phase_start = time.perf_counter()
        ranked = rank_sites(deduped, params)
        phase_timings["rank_s"] = time.perf_counter() - phase_start

        phase_start = time.perf_counter()
        truncated = self._truncate_sites(ranked, params.top_n_sites)
        phase_timings["truncate_s"] = time.perf_counter() - phase_start

        total_s = time.perf_counter() - started_at
        return {
            "sites": truncated,
            "phase_timings": phase_timings,
            "total_s": total_s,
            "resolved_fasta": str(fasta_path),
            "resolved_annotation": resolved_annotation,
            "shard_count": len(shard_specs),
            "effective_workers": effective_workers,
            "raw_sites": len(raw_sites),
            "deduped_sites": len(deduped),
            "ranked_sites": len(ranked),
            "retained_sites": len(truncated),
        }


def run_backend_case(
    case: BenchmarkCase,
    *,
    backend: ZFNSearchBackend,
    repeat: int,
    baseline_frame: pd.DataFrame | None = None,
    search_space_index: str | None = None,
) -> tuple[BenchmarkRun, pd.DataFrame]:
    params = make_params(case=case, backend=backend, search_space_index=search_space_index)
    annotation = make_annotation(case)
    result = InstrumentedSearcher().profile(params, annotation=annotation)
    normalized_frame = sites_to_frame(result["sites"])
    fingerprint = fingerprint_frame(normalized_frame)
    parity_ok = True if baseline_frame is None else normalized_frame.equals(baseline_frame)
    run = BenchmarkRun(
        case_id=case.case_id,
        case_label=case.label,
        backend=backend.value,
        repeat=repeat,
        annotation_enabled=annotation is not None,
        wall_s=result["total_s"],
        result_rows=int(len(normalized_frame)),
        parity_ok=parity_ok,
        fingerprint=fingerprint,
        resolved_fasta=result["resolved_fasta"],
        resolved_annotation=result["resolved_annotation"],
        shard_count=result["shard_count"],
        effective_workers=result["effective_workers"],
        raw_sites=result["raw_sites"],
        deduped_sites=result["deduped_sites"],
        ranked_sites=result["ranked_sites"],
        retained_sites=result["retained_sites"],
        resolve_inputs_s=result["phase_timings"]["resolve_inputs_s"],
        load_fasta_s=result["phase_timings"]["load_fasta_s"],
        build_shards_s=result["phase_timings"]["build_shards_s"],
        search_shards_s=result["phase_timings"]["search_shards_s"],
        dedupe_s=result["phase_timings"]["dedupe_s"],
        annotate_s=result["phase_timings"]["annotate_s"],
        rank_s=result["phase_timings"]["rank_s"],
        truncate_s=result["phase_timings"]["truncate_s"],
    )
    return run, normalized_frame


def run_benchmark_suite(case: BenchmarkCase) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    rows: list[dict[str, Any]] = []
    normalized_frames: dict[str, pd.DataFrame] = {}

    baseline_frame: pd.DataFrame | None = None
    for backend in LIVE_BACKENDS:
        for repeat in range(1, case.repeats + 1):
            print(f"[{case.case_id}] backend={backend.value} repeat={repeat}/{case.repeats}")
            run, normalized_frame = run_backend_case(
                case,
                backend=backend,
                repeat=repeat,
                baseline_frame=baseline_frame,
            )
            rows.append(asdict(run))
            normalized_frames[f"{backend.value}_repeat_{repeat}"] = normalized_frame
            if backend == ZFNSearchBackend.EXHAUSTIVE_PYTHON and repeat == 1:
                baseline_frame = normalized_frame
                rows[-1]["parity_ok"] = True

    results = pd.DataFrame(rows)
    return results, normalized_frames


def run_all_cases(cases: list[BenchmarkCase]) -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    all_results: list[pd.DataFrame] = []
    all_frames: dict[str, pd.DataFrame] = {}
    for case in cases:
        case_results, case_frames = run_benchmark_suite(case)
        all_results.append(case_results)
        for frame_key, frame in case_frames.items():
            all_frames[f"{case.case_id}::{frame_key}"] = frame
    return pd.concat(all_results, ignore_index=True), all_frames


def plot_runtime_results(results: pd.DataFrame) -> None:
    case_order = results["case_label"].drop_duplicates().tolist()
    fig, axes = plt.subplots(len(case_order), 2, figsize=(14, max(5, 5 * len(case_order))), squeeze=False)

    for row_index, case_label in enumerate(case_order):
        case_results = results.loc[results["case_label"] == case_label].copy()
        runtime_ax = axes[row_index][0]
        phase_ax = axes[row_index][1]

        sns.barplot(data=case_results, x="backend", y="wall_s", errorbar="sd", ax=runtime_ax)
        runtime_ax.set_title(f"Wall-clock runtime: {case_label}")
        runtime_ax.set_ylabel("Wall time (s)")
        runtime_ax.set_xlabel("Backend")
        runtime_ax.tick_params(axis="x", rotation=15)

        phase_frame = (
            case_results.groupby("backend", as_index=False)[PHASE_COLUMNS]
            .mean()
            .melt(id_vars="backend", var_name="phase", value_name="seconds")
        )
        sns.barplot(data=phase_frame, x="phase", y="seconds", hue="backend", ax=phase_ax)
        phase_ax.set_title(f"Mean phase timings: {case_label}")
        phase_ax.set_ylabel("Seconds")
        phase_ax.set_xlabel("Phase")
        phase_ax.tick_params(axis="x", rotation=35)

    plt.tight_layout()
    plt.show()

## 6. Run a Minimal End-to-End Example

Execute all enabled cases. By default this runs only the deterministic fixture; if `RUN_REAL_CHR3` is enabled it will also benchmark real Ensembl chr3 with GTF-backed annotation.

In [ ]:
results_df, normalized_frames = run_all_cases(CASES)
summary_df = summarize_results(results_df)

parity_rows: list[dict[str, Any]] = []
for case in CASES:
    baseline_key = f"{case.case_id}::{ZFNSearchBackend.EXHAUSTIVE_PYTHON.value}_repeat_1"
    baseline_frame = normalized_frames[baseline_key]
    for backend in LIVE_BACKENDS:
        for repeat in range(1, case.repeats + 1):
            frame_key = f"{case.case_id}::{backend.value}_repeat_{repeat}"
            frame = normalized_frames[frame_key]
            parity_rows.append(
                {
                    "case_id": case.case_id,
                    "case_label": case.label,
                    "backend": backend.value,
                    "repeat": repeat,
                    "rows": len(frame),
                    "fingerprint": fingerprint_frame(frame),
                    "matches_exhaustive_repeat_1": frame.equals(baseline_frame),
                }
            )
parity_summary_df = pd.DataFrame(parity_rows)
annotation_summary_df = summarize_annotation_regions(normalized_frames, CASES)

display(results_df)
display(summary_df)
display(parity_summary_df)
if not annotation_summary_df.empty:
    display(annotation_summary_df)
plot_runtime_results(results_df)

## 7. Add Basic Validation Checks

Assert that every live backend matches the exhaustive baseline on normalized output rows and that the timing/result columns are populated.

In [ ]:
assert not results_df.empty
for case in CASES:
    case_results = results_df.loc[results_df["case_id"] == case.case_id]
    assert set(case_results["backend"]) == {backend.value for backend in LIVE_BACKENDS}
    assert case_results["parity_ok"].all(), f"A backend diverged from the exhaustive baseline for {case.case_id}."
    assert case_results["result_rows"].gt(0).all()
    for column in ["wall_s", *PHASE_COLUMNS]:
        assert case_results[column].notna().all(), f"Missing values in {column} for {case.case_id}"
        assert case_results[column].ge(0).all(), f"Negative timings in {column} for {case.case_id}"
    if case.annotation_reference is not None:
        assert case_results["annotate_s"].gt(0).any(), f"Annotation timing was never populated for {case.case_id}"

assert parity_summary_df["matches_exhaustive_repeat_1"].all()
for case in CASES:
    case_parity = parity_summary_df.loc[parity_summary_df["case_id"] == case.case_id]
    reference_fingerprint = case_parity.loc[
        case_parity["backend"] == ZFNSearchBackend.EXHAUSTIVE_PYTHON.value, "fingerprint"
    ].iloc[0]
    comparison_fingerprints = case_parity.groupby("backend")["fingerprint"].first().to_dict()
    assert comparison_fingerprints[ZFNSearchBackend.PYAHOCORASICK.value] == reference_fingerprint
    assert comparison_fingerprints[ZFNSearchBackend.FM_INDEX.value] == reference_fingerprint

print("Validation checks passed.")

## 8. Persist Output Artifacts

Write detailed result tables, summary tables, parity summaries, and a small run manifest to disk for reuse outside the notebook.

In [ ]:
results_path = ARTIFACT_ROOT / "backend_runtime_results.csv"
summary_path = ARTIFACT_ROOT / "backend_runtime_summary.csv"
parity_path = ARTIFACT_ROOT / "backend_runtime_parity.csv"
annotation_summary_path = ARTIFACT_ROOT / "backend_runtime_annotation_regions.csv"
manifest_path = ARTIFACT_ROOT / "backend_runtime_manifest.json"

results_df.to_csv(results_path, index=False)
summary_df.to_csv(summary_path, index=False)
parity_summary_df.to_csv(parity_path, index=False)
if not annotation_summary_df.empty:
    annotation_summary_df.to_csv(annotation_summary_path, index=False)
manifest = {
    "cases": [
        {
            **asdict(case),
            "search_space_fasta": str(case.search_space_fasta),
        }
        for case in CASES
    ],
    "backends": [backend.value for backend in LIVE_BACKENDS],
    "phase_columns": PHASE_COLUMNS,
    "artifacts": {
        "results": str(results_path),
        "summary": str(summary_path),
        "parity": str(parity_path),
        "annotation_summary": str(annotation_summary_path) if not annotation_summary_df.empty else None,
    },
}
manifest_json = json.dumps(manifest, indent=2, sort_keys=True)
manifest_path.write_text(manifest_json, encoding="utf-8")

print(manifest_json)

## Appendix: FM-index Persisted Bundle Check

Optionally build a persisted FM-index bundle and compare it against the live FM-index run on the same deterministic fixture. This stays separate from the core three-backend chart.

In [ ]:
appendix_case = CASES[0]
appendix_baseline = normalized_frames[f"{appendix_case.case_id}::{ZFNSearchBackend.EXHAUSTIVE_PYTHON.value}_repeat_1"]
if RUN_FM_INDEX_PERSISTED_APPENDIX:
    bundle_dir = ARTIFACT_ROOT / "fm_index_bundle"
    bundle_summary = build_zfn_search_index(
        backend=ZFNSearchBackend.FM_INDEX,
        genome_fasta=Path(appendix_case.search_space_fasta),
        output_dir=bundle_dir,
    )
    persisted_run, persisted_frame = run_backend_case(
        appendix_case,
        backend=ZFNSearchBackend.FM_INDEX,
        repeat=1,
        baseline_frame=appendix_baseline,
        search_space_index=str(bundle_dir),
    )
    persisted_summary_df = pd.DataFrame([asdict(persisted_run)])
    persisted_summary_df["bundle_dir"] = str(bundle_dir)
    persisted_summary_df["bundle_artifact"] = bundle_summary["artifact"]
    persisted_summary_df["matches_live_fm_index"] = persisted_frame.equals(
        normalized_frames[f"{appendix_case.case_id}::{ZFNSearchBackend.FM_INDEX.value}_repeat_1"]
    )
    persisted_summary_df.to_csv(ARTIFACT_ROOT / "backend_runtime_fm_index_persisted.csv", index=False)
    display(persisted_summary_df)
else:
    print("Persisted FM-index appendix skipped.")

## Appendix: Fragmentation, Core, and Memory Sweet Spots

Profile the two library-backed search engines under a controlled fragmentation sweep. This section keeps total base pairs fixed while varying contig count, then separates shared FASTA parsing from backend-specific preparation and warmed shard scanning.

The goal is to show where each backend sits on the runtime versus memory frontier as fragmentation and worker counts change.

In [ ]:
import gc
import threading

import psutil

FRAGMENTATION_BACKENDS = [
    ZFNSearchBackend.PYAHOCORASICK,
    ZFNSearchBackend.FM_INDEX,
]
FRAGMENTATION_TOTAL_BP = 2_400_000
FRAGMENTATION_CONTIG_COUNTS = [1, 4, 16, 64, 256]
FRAGMENTATION_WORKER_COUNTS = [1, 2, 4]
FRAGMENTATION_REPEATS = 1
FRAGMENTATION_TOP_N_SITES = TOP_N_SITES
FRAGMENTATION_FIXTURE_ROOT = ARTIFACT_ROOT / "fragmentation_fixtures"
FRAGMENTATION_FIXTURE_ROOT.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class FragmentationProfileRun:
    backend: str
    fragment_count: int
    requested_workers: int
    repeat: int
    total_bp: int
    bp_per_fragment: int
    shard_count: int
    effective_workers: int
    load_fasta_s: float
    build_shards_s: float
    backend_prepare_s: float
    search_shards_warm_s: float
    approx_cold_wall_s: float
    load_rss_delta_mb: float
    backend_prepare_rss_delta_mb: float
    search_rss_delta_mb: float
    peak_rss_mb: float
    load_cpu_s: float
    backend_prepare_cpu_s: float
    search_cpu_s: float
    search_core_equiv: float
    raw_sites: int


class PeakRSSMonitor:
    def __init__(self, process: psutil.Process, sample_interval_s: float = 0.01) -> None:
        self.process = process
        self.sample_interval_s = sample_interval_s
        self._stop = threading.Event()
        self._thread: threading.Thread | None = None
        self.peak_rss_mb = self._rss_mb()

    def _rss_mb(self) -> float:
        return self.process.memory_info().rss / (1024**2)

    def _run(self) -> None:
        while not self._stop.wait(self.sample_interval_s):
            self.peak_rss_mb = max(self.peak_rss_mb, self._rss_mb())

    def __enter__(self) -> "PeakRSSMonitor":
        self.peak_rss_mb = self._rss_mb()
        self._stop.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()
        return self

    def __exit__(self, exc_type, exc, tb) -> None:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=0.5)
        self.peak_rss_mb = max(self.peak_rss_mb, self._rss_mb())


def repeated_sequence(motif: str, length: int) -> str:
    repeats = (length // len(motif)) + 1
    return (motif * repeats)[:length]


def build_fragmented_fixture(total_bp: int, fragment_count: int, output_dir: Path) -> Path:
    site_templates = [
        canonical_site(LEFT_HALF_SITE, RIGHT_HALF_SITE, spacer="AAAAA"),
        canonical_site(LEFT_HALF_SITE, RIGHT_HALF_SITE, spacer="AAAAAA"),
    ]
    base_length = total_bp // fragment_count
    remainder = total_bp % fragment_count
    fasta_path = output_dir / f"fragmented_{total_bp}bp_{fragment_count:03d}_contigs.fa"
    motifs = ["ACGT", "TGCA", "GATC", "CATG", "TTAA", "CGCG"]

    lines: list[str] = []
    for index in range(fragment_count):
        contig_len = base_length + (1 if index < remainder else 0)
        site = site_templates[index % len(site_templates)]
        if contig_len <= len(site) + 8:
            raise ValueError(f"Contig length {contig_len} is too small for embedded site profiling")
        prefix_len = (contig_len - len(site)) // 2
        suffix_len = contig_len - len(site) - prefix_len
        left_motif = motifs[index % len(motifs)]
        right_motif = motifs[(index + 2) % len(motifs)]
        sequence = repeated_sequence(left_motif, prefix_len) + site + repeated_sequence(right_motif, suffix_len)
        lines.append(f">frag_{index + 1:03d}")
        lines.append(sequence)

    fasta_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return fasta_path


def make_fragmentation_params(
    *,
    fasta_path: Path,
    backend: ZFNSearchBackend,
    requested_workers: int,
) -> ZFNDesignParameters:
    return ZFNDesignParameters(
        search_space_fasta=str(fasta_path),
        left_half_site=LEFT_HALF_SITE,
        right_half_site=RIGHT_HALF_SITE,
        search_backend=backend,
        algorithm=ZFNAlgorithm.HOMOLOGY,
        top_n_sites=FRAGMENTATION_TOP_N_SITES,
        half_site_constraints=ZFNHalfSiteConstraints(
            max_mismatches=0,
            seed_len_from_fokI=6,
            seed_max_mismatches=0,
            window_stride=1,
        ),
        spacer_constraints=ZFNSpacerConstraints(allowed_spacer_lengths=ALLOWED_SPACERS),
        sharding=ZFNShardingConfig(
            enabled=True,
            chunk_size_bp=FRAGMENTATION_TOTAL_BP + 1,
            overlap_bp=50,
            chromosomes=[],
            max_workers=requested_workers,
        ),
    )


def process_cpu_s(process: psutil.Process) -> float:
    times = process.cpu_times()
    return float(times.user + times.system)


def prime_backend_state(
    *,
    searcher: ExhaustiveZFNOffTargetSearcher,
    chrom_sequences: dict[str, str],
    params: ZFNDesignParameters,
) -> None:
    scan_engine = searcher._scan_engine_for(params)
    if params.search_backend == ZFNSearchBackend.PYAHOCORASICK:
        scan_engine._automaton("left", params.left_half_site, params)
        scan_engine._automaton("right", params.right_half_site, params)
        return
    if params.search_backend == ZFNSearchBackend.FM_INDEX:
        scan_engine._live_multi_index(chrom_sequences)
        return
    raise ValueError(f"Unsupported fragmentation profiler backend: {params.search_backend}")


def profile_fragmentation_case(
    *,
    fasta_path: Path,
    fragment_count: int,
    backend: ZFNSearchBackend,
    requested_workers: int,
    repeat: int,
) -> FragmentationProfileRun:
    gc.collect()
    process = psutil.Process()
    searcher = ExhaustiveZFNOffTargetSearcher()
    params = make_fragmentation_params(
        fasta_path=fasta_path,
        backend=backend,
        requested_workers=requested_workers,
    )

    baseline_rss_mb = process.memory_info().rss / (1024**2)

    cpu_start = process_cpu_s(process)
    wall_start = time.perf_counter()
    chrom_sequences = searcher._load_fasta(Path(params.search_space_fasta))
    load_fasta_s = time.perf_counter() - wall_start
    load_cpu_s = process_cpu_s(process) - cpu_start
    rss_after_load_mb = process.memory_info().rss / (1024**2)

    wall_start = time.perf_counter()
    shard_specs = searcher._build_shard_specs(chrom_sequences, params)
    build_shards_s = time.perf_counter() - wall_start

    cpu_start = process_cpu_s(process)
    wall_start = time.perf_counter()
    with PeakRSSMonitor(process) as monitor:
        prime_backend_state(searcher=searcher, chrom_sequences=chrom_sequences, params=params)
    backend_prepare_s = time.perf_counter() - wall_start
    backend_prepare_cpu_s = process_cpu_s(process) - cpu_start
    rss_after_prepare_mb = process.memory_info().rss / (1024**2)
    prepare_peak_rss_mb = monitor.peak_rss_mb

    effective_workers = (
        min(searcher._recommended_worker_cap(chrom_sequences, params), len(shard_specs)) if shard_specs else 1
    )
    scan_engine = searcher._scan_engine_for(params)

    cpu_start = process_cpu_s(process)
    wall_start = time.perf_counter()
    with PeakRSSMonitor(process) as monitor:
        raw_sites = searcher._run_shard_searches(
            shard_specs=shard_specs,
            chrom_sequences=chrom_sequences,
            params=params,
            scan_engine=scan_engine,
            workers=effective_workers,
            started_at=wall_start,
        )
    search_shards_warm_s = time.perf_counter() - wall_start
    search_cpu_s = process_cpu_s(process) - cpu_start
    rss_after_search_mb = process.memory_info().rss / (1024**2)
    search_peak_rss_mb = monitor.peak_rss_mb

    peak_rss_mb = max(
        baseline_rss_mb,
        rss_after_load_mb,
        rss_after_prepare_mb,
        rss_after_search_mb,
        prepare_peak_rss_mb,
        search_peak_rss_mb,
    )
    search_core_equiv = search_cpu_s / search_shards_warm_s if search_shards_warm_s > 0 else 0.0

    return FragmentationProfileRun(
        backend=backend.value,
        fragment_count=fragment_count,
        requested_workers=requested_workers,
        repeat=repeat,
        total_bp=FRAGMENTATION_TOTAL_BP,
        bp_per_fragment=FRAGMENTATION_TOTAL_BP // fragment_count,
        shard_count=len(shard_specs),
        effective_workers=effective_workers,
        load_fasta_s=load_fasta_s,
        build_shards_s=build_shards_s,
        backend_prepare_s=backend_prepare_s,
        search_shards_warm_s=search_shards_warm_s,
        approx_cold_wall_s=load_fasta_s + build_shards_s + backend_prepare_s + search_shards_warm_s,
        load_rss_delta_mb=max(0.0, rss_after_load_mb - baseline_rss_mb),
        backend_prepare_rss_delta_mb=max(0.0, rss_after_prepare_mb - rss_after_load_mb),
        search_rss_delta_mb=max(0.0, rss_after_search_mb - rss_after_prepare_mb),
        peak_rss_mb=peak_rss_mb,
        load_cpu_s=load_cpu_s,
        backend_prepare_cpu_s=backend_prepare_cpu_s,
        search_cpu_s=search_cpu_s,
        search_core_equiv=search_core_equiv,
        raw_sites=len(raw_sites),
    )


def run_fragmentation_profile() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for fragment_count in FRAGMENTATION_CONTIG_COUNTS:
        fasta_path = build_fragmented_fixture(
            total_bp=FRAGMENTATION_TOTAL_BP,
            fragment_count=fragment_count,
            output_dir=FRAGMENTATION_FIXTURE_ROOT,
        )
        for backend in FRAGMENTATION_BACKENDS:
            for requested_workers in FRAGMENTATION_WORKER_COUNTS:
                for repeat in range(1, FRAGMENTATION_REPEATS + 1):
                    print(
                        f"fragmentation={fragment_count} backend={backend.value} "
                        f"workers={requested_workers} repeat={repeat}/{FRAGMENTATION_REPEATS}"
                    )
                    run = profile_fragmentation_case(
                        fasta_path=fasta_path,
                        fragment_count=fragment_count,
                        backend=backend,
                        requested_workers=requested_workers,
                        repeat=repeat,
                    )
                    rows.append(asdict(run))
                    gc.collect()
    return pd.DataFrame(rows)


def summarize_fragmentation_profile(results: pd.DataFrame) -> pd.DataFrame:
    return (
        results.groupby(["backend", "fragment_count", "requested_workers"], as_index=False)
        .agg(
            mean_load_fasta_s=("load_fasta_s", "mean"),
            mean_build_shards_s=("build_shards_s", "mean"),
            mean_backend_prepare_s=("backend_prepare_s", "mean"),
            mean_search_shards_warm_s=("search_shards_warm_s", "mean"),
            mean_approx_cold_wall_s=("approx_cold_wall_s", "mean"),
            mean_peak_rss_mb=("peak_rss_mb", "mean"),
            mean_search_core_equiv=("search_core_equiv", "mean"),
            mean_effective_workers=("effective_workers", "mean"),
            mean_raw_sites=("raw_sites", "mean"),
        )
        .sort_values(["backend", "fragment_count", "requested_workers"])
        .reset_index(drop=True)
    )


def pareto_frontier(summary: pd.DataFrame) -> pd.DataFrame:
    frontier_rows: list[pd.Series] = []
    for backend, backend_frame in summary.groupby("backend"):
        backend_frame = backend_frame.reset_index(drop=True)
        for row_index, row in backend_frame.iterrows():
            dominated = backend_frame.loc[
                (backend_frame.index != row_index)
                & (backend_frame["mean_approx_cold_wall_s"] <= row["mean_approx_cold_wall_s"])
                & (backend_frame["mean_peak_rss_mb"] <= row["mean_peak_rss_mb"])
                & (
                    (backend_frame["mean_approx_cold_wall_s"] < row["mean_approx_cold_wall_s"])
                    | (backend_frame["mean_peak_rss_mb"] < row["mean_peak_rss_mb"])
                )
            ]
            if dominated.empty:
                frontier_rows.append(row)
    if not frontier_rows:
        return pd.DataFrame(columns=summary.columns)
    return pd.DataFrame(frontier_rows).sort_values(["backend", "mean_peak_rss_mb", "mean_approx_cold_wall_s"])


def sweet_spot_recommendations(summary: pd.DataFrame) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for backend, backend_frame in summary.groupby("backend"):
        fastest = backend_frame.sort_values(["mean_approx_cold_wall_s", "mean_peak_rss_mb"]).iloc[0]
        leanest = backend_frame.sort_values(["mean_peak_rss_mb", "mean_approx_cold_wall_s"]).iloc[0]
        balanced = (
            backend_frame.assign(
                balance_rank=(
                    backend_frame["mean_approx_cold_wall_s"] / backend_frame["mean_approx_cold_wall_s"].min()
                    + backend_frame["mean_peak_rss_mb"] / backend_frame["mean_peak_rss_mb"].min()
                )
            )
            .sort_values(["balance_rank", "mean_approx_cold_wall_s"])
            .iloc[0]
        )
        for profile_name, row in [("fastest", fastest), ("leanest", leanest), ("balanced", balanced)]:
            rows.append(
                {
                    "backend": backend,
                    "profile": profile_name,
                    "fragment_count": int(row["fragment_count"]),
                    "requested_workers": int(row["requested_workers"]),
                    "mean_approx_cold_wall_s": float(row["mean_approx_cold_wall_s"]),
                    "mean_peak_rss_mb": float(row["mean_peak_rss_mb"]),
                    "mean_search_core_equiv": float(row["mean_search_core_equiv"]),
                }
            )
    return pd.DataFrame(rows)


def plot_fragmentation_profile(summary: pd.DataFrame, frontier: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    load_ax, prepare_ax, wall_ax, frontier_ax = axes.flatten()

    sns.lineplot(
        data=summary,
        x="fragment_count",
        y="mean_load_fasta_s",
        hue="backend",
        style="requested_workers",
        markers=True,
        dashes=False,
        ax=load_ax,
    )
    load_ax.set_xscale("log", base=2)
    load_ax.set_title("Shared FASTA parse cost versus fragmentation")
    load_ax.set_ylabel("Seconds")
    load_ax.set_xlabel("Contig count")

    sns.lineplot(
        data=summary,
        x="fragment_count",
        y="mean_backend_prepare_s",
        hue="backend",
        style="requested_workers",
        markers=True,
        dashes=False,
        ax=prepare_ax,
    )
    prepare_ax.set_xscale("log", base=2)
    prepare_ax.set_title("Backend-specific prepare cost")
    prepare_ax.set_ylabel("Seconds")
    prepare_ax.set_xlabel("Contig count")

    sns.lineplot(
        data=summary,
        x="fragment_count",
        y="mean_approx_cold_wall_s",
        hue="backend",
        style="requested_workers",
        markers=True,
        dashes=False,
        ax=wall_ax,
    )
    wall_ax.set_xscale("log", base=2)
    wall_ax.set_title("Approximate cold runtime")
    wall_ax.set_ylabel("Seconds")
    wall_ax.set_xlabel("Contig count")

    sns.scatterplot(
        data=summary,
        x="mean_peak_rss_mb",
        y="mean_approx_cold_wall_s",
        hue="backend",
        style="requested_workers",
        size="fragment_count",
        sizes=(60, 220),
        ax=frontier_ax,
    )
    if not frontier.empty:
        sns.scatterplot(
            data=frontier,
            x="mean_peak_rss_mb",
            y="mean_approx_cold_wall_s",
            hue="backend",
            marker="X",
            s=250,
            legend=False,
            ax=frontier_ax,
        )
    frontier_ax.set_title("Runtime versus memory frontier")
    frontier_ax.set_xlabel("Peak RSS (MB)")
    frontier_ax.set_ylabel("Approximate cold runtime (s)")

    plt.tight_layout()
    plt.show()

In [ ]:
fragmentation_results_df = run_fragmentation_profile()
fragmentation_summary_df = summarize_fragmentation_profile(fragmentation_results_df)
fragmentation_frontier_df = pareto_frontier(fragmentation_summary_df)
fragmentation_sweet_spots_df = sweet_spot_recommendations(fragmentation_summary_df)

fragmentation_results_path = ARTIFACT_ROOT / "backend_fragmentation_profile_results.csv"
fragmentation_summary_path = ARTIFACT_ROOT / "backend_fragmentation_profile_summary.csv"
fragmentation_frontier_path = ARTIFACT_ROOT / "backend_fragmentation_profile_frontier.csv"
fragmentation_sweet_spots_path = ARTIFACT_ROOT / "backend_fragmentation_profile_sweet_spots.csv"

fragmentation_results_df.to_csv(fragmentation_results_path, index=False)
fragmentation_summary_df.to_csv(fragmentation_summary_path, index=False)
fragmentation_frontier_df.to_csv(fragmentation_frontier_path, index=False)
fragmentation_sweet_spots_df.to_csv(fragmentation_sweet_spots_path, index=False)

display(fragmentation_results_df)
display(fragmentation_summary_df)
display(fragmentation_frontier_df)
display(fragmentation_sweet_spots_df)
plot_fragmentation_profile(fragmentation_summary_df, fragmentation_frontier_df)

print("Saved fragmentation profiling artifacts:")
print(fragmentation_results_path)
print(fragmentation_summary_path)
print(fragmentation_frontier_path)
print(fragmentation_sweet_spots_path)

In [ ]:
REAL_CHR3_SLICE_TOTAL_BP = 2_400_000
REAL_CHR3_SLICE_COUNTS = [1, 4, 16, 64]
REAL_CHR3_SLICE_WORKER_COUNTS = [1, 2, 4]
REAL_CHR3_SLICE_REPEATS = 1
REAL_CHR3_SLICE_FIXTURE_ROOT = ARTIFACT_ROOT / "real_chr3_slice_fixtures"
REAL_CHR3_SLICE_FIXTURE_ROOT.mkdir(parents=True, exist_ok=True)


def resolve_real_chr3_fasta_path() -> Path:
    params = ZFNDesignParameters(
        search_space_fasta=HG38_CHR3_FASTA_URL,
        left_half_site=LEFT_HALF_SITE,
        right_half_site=RIGHT_HALF_SITE,
        search_backend=ZFNSearchBackend.PYAHOCORASICK,
        algorithm=ZFNAlgorithm.HOMOLOGY,
        top_n_sites=FRAGMENTATION_TOP_N_SITES,
        half_site_constraints=ZFNHalfSiteConstraints(
            max_mismatches=0,
            seed_len_from_fokI=6,
            seed_max_mismatches=0,
            window_stride=1,
        ),
        spacer_constraints=ZFNSpacerConstraints(allowed_spacer_lengths=ALLOWED_SPACERS),
        sharding=ZFNShardingConfig(
            enabled=True, chunk_size_bp=REAL_CHR3_SLICE_TOTAL_BP + 1, overlap_bp=50, chromosomes=[], max_workers=1
        ),
    )
    return ExhaustiveZFNOffTargetSearcher()._resolve_search_space_fasta(params)


def load_single_sequence_fasta(fasta_path: Path) -> str:
    chrom_sequences = ExhaustiveZFNOffTargetSearcher()._load_fasta(fasta_path)
    if len(chrom_sequences) != 1:
        raise ValueError(f"Expected one contig in hg38 chr3 FASTA, found {len(chrom_sequences)}")
    return next(iter(chrom_sequences.values()))


def build_real_chr3_slice_fixture(
    reference_sequence: str, total_bp: int, fragment_count: int, output_dir: Path
) -> Path:
    base_length = total_bp // fragment_count
    remainder = total_bp % fragment_count
    usable_length = sum(base_length + (1 if idx < remainder else 0) for idx in range(fragment_count))
    if usable_length > len(reference_sequence):
        raise ValueError(f"Requested {usable_length} bp from chr3, but only {len(reference_sequence)} bp available")

    span = len(reference_sequence) - usable_length
    start_positions = (
        [0] if fragment_count == 1 else [int(round(span * idx / (fragment_count - 1))) for idx in range(fragment_count)]
    )
    fasta_path = output_dir / f"real_chr3_slices_{total_bp}bp_{fragment_count:03d}_contigs.fa"

    lines: list[str] = []
    for idx, start in enumerate(start_positions):
        contig_len = base_length + (1 if idx < remainder else 0)
        sequence = reference_sequence[start : start + contig_len]
        if len(sequence) != contig_len:
            raise ValueError(f"Slice {idx} produced {len(sequence)} bp, expected {contig_len}")
        lines.append(f">chr3_slice_{idx + 1:03d}")
        lines.append(sequence)

    fasta_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return fasta_path


def run_real_chr3_slice_profile(reference_sequence: str) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for fragment_count in REAL_CHR3_SLICE_COUNTS:
        fasta_path = build_real_chr3_slice_fixture(
            reference_sequence=reference_sequence,
            total_bp=REAL_CHR3_SLICE_TOTAL_BP,
            fragment_count=fragment_count,
            output_dir=REAL_CHR3_SLICE_FIXTURE_ROOT,
        )
        for backend in FRAGMENTATION_BACKENDS:
            for requested_workers in REAL_CHR3_SLICE_WORKER_COUNTS:
                for repeat in range(1, REAL_CHR3_SLICE_REPEATS + 1):
                    print(
                        f"real_chr3 fragmentation={fragment_count} backend={backend.value} "
                        f"workers={requested_workers} repeat={repeat}/{REAL_CHR3_SLICE_REPEATS}"
                    )
                    run = profile_fragmentation_case(
                        fasta_path=fasta_path,
                        fragment_count=fragment_count,
                        backend=backend,
                        requested_workers=requested_workers,
                        repeat=repeat,
                    )
                    payload = asdict(run)
                    payload["dataset"] = "real_chr3_slices"
                    rows.append(payload)
                    gc.collect()
    return pd.DataFrame(rows)


def compare_synthetic_vs_real_sweet_spots(
    synthetic_summary: pd.DataFrame,
    real_summary: pd.DataFrame,
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for backend in sorted(set(synthetic_summary["backend"]).intersection(real_summary["backend"])):
        synthetic_fastest = (
            synthetic_summary.loc[synthetic_summary["backend"] == backend]
            .sort_values(["mean_approx_cold_wall_s", "mean_peak_rss_mb"])
            .iloc[0]
        )
        real_fastest = (
            real_summary.loc[real_summary["backend"] == backend]
            .sort_values(["mean_approx_cold_wall_s", "mean_peak_rss_mb"])
            .iloc[0]
        )
        rows.append(
            {
                "backend": backend,
                "synthetic_fragment_count": int(synthetic_fastest["fragment_count"]),
                "synthetic_requested_workers": int(synthetic_fastest["requested_workers"]),
                "real_fragment_count": int(real_fastest["fragment_count"]),
                "real_requested_workers": int(real_fastest["requested_workers"]),
                "sweet_spot_matches": (
                    int(synthetic_fastest["fragment_count"]) == int(real_fastest["fragment_count"])
                    and int(synthetic_fastest["requested_workers"]) == int(real_fastest["requested_workers"])
                ),
                "synthetic_mean_approx_cold_wall_s": float(synthetic_fastest["mean_approx_cold_wall_s"]),
                "real_mean_approx_cold_wall_s": float(real_fastest["mean_approx_cold_wall_s"]),
                "synthetic_mean_peak_rss_mb": float(synthetic_fastest["mean_peak_rss_mb"]),
                "real_mean_peak_rss_mb": float(real_fastest["mean_peak_rss_mb"]),
            }
        )
    return pd.DataFrame(rows)


def plot_real_chr3_slice_profile(summary: pd.DataFrame, frontier: pd.DataFrame) -> None:
    plot_fragmentation_profile(summary, frontier)

## Appendix: Real hg38 chr3 Slice Sweep

Repeat the fragmentation sweep using slices sampled from real hg38 chr3 sequence. This keeps the benchmarking contract aligned with the synthetic sweep while testing whether backend preferences still hold under biologically realistic composition.

In [ ]:
real_chr3_fasta_path = resolve_real_chr3_fasta_path()
real_chr3_sequence = load_single_sequence_fasta(real_chr3_fasta_path)
real_chr3_slice_results_df = run_real_chr3_slice_profile(real_chr3_sequence)
real_chr3_slice_summary_df = summarize_fragmentation_profile(real_chr3_slice_results_df)
real_chr3_slice_frontier_df = pareto_frontier(real_chr3_slice_summary_df)
real_chr3_slice_sweet_spots_df = sweet_spot_recommendations(real_chr3_slice_summary_df)
real_chr3_vs_synthetic_df = compare_synthetic_vs_real_sweet_spots(
    fragmentation_summary_df,
    real_chr3_slice_summary_df,
)

real_chr3_slice_results_path = ARTIFACT_ROOT / "backend_real_chr3_slice_profile_results.csv"
real_chr3_slice_summary_path = ARTIFACT_ROOT / "backend_real_chr3_slice_profile_summary.csv"
real_chr3_slice_frontier_path = ARTIFACT_ROOT / "backend_real_chr3_slice_profile_frontier.csv"
real_chr3_slice_sweet_spots_path = ARTIFACT_ROOT / "backend_real_chr3_slice_profile_sweet_spots.csv"
real_chr3_vs_synthetic_path = ARTIFACT_ROOT / "backend_real_chr3_slice_vs_synthetic.csv"

real_chr3_slice_results_df.to_csv(real_chr3_slice_results_path, index=False)
real_chr3_slice_summary_df.to_csv(real_chr3_slice_summary_path, index=False)
real_chr3_slice_frontier_df.to_csv(real_chr3_slice_frontier_path, index=False)
real_chr3_slice_sweet_spots_df.to_csv(real_chr3_slice_sweet_spots_path, index=False)
real_chr3_vs_synthetic_df.to_csv(real_chr3_vs_synthetic_path, index=False)

display(real_chr3_slice_summary_df)
display(real_chr3_slice_frontier_df)
display(real_chr3_slice_sweet_spots_df)
display(real_chr3_vs_synthetic_df)
plot_real_chr3_slice_profile(real_chr3_slice_summary_df, real_chr3_slice_frontier_df)

print("Saved real chr3 slice profiling artifacts:")
print(real_chr3_slice_results_path)
print(real_chr3_slice_summary_path)
print(real_chr3_slice_frontier_path)
print(real_chr3_slice_sweet_spots_path)
print(real_chr3_vs_synthetic_path)